# Vesuvius Visualization: Model Output vs Ground Truth

Runs inference on a training sample and generates:
1. Probability heatmaps
2. Before/after post-processing comparison
3. Diff maps (what PP changed)
4. 3D surface renders

In [ ]:
from IPython.display import clear_output

var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
  "$var"/keras_nightly-*.whl \
  "$var"/tifffile-*.whl \
  "$var"/imagecodecs-*.whl \
  "$var"/medicai-*.whl \
  --no-index \
  --find-links "$var"

clear_output()

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from keras import ops
from medicai.transforms import Compose, NormalizeIntensity
from medicai.models import TransUNet
from medicai.utils.inference import SlidingWindowInference

import numpy as np
import pandas as pd
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects
from skimage import measure
from scipy.ndimage import zoom
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

## Load Model + Pick a Training Sample

In [ ]:
root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
train_img_dir = f"{root_dir}/train_images"
train_lbl_dir = f"{root_dir}/train_labels"
out_root = "/kaggle/working/visualizations"
os.makedirs(out_root, exist_ok=True)

train_df = pd.read_csv(f"{root_dir}/train.csv")
print(f"Training samples: {len(train_df)}")
train_df.head()

In [ ]:
# Pick a training sample that actually exists on disk
sample_id = None
for vid in train_df["id"]:
    if os.path.exists(f"{train_img_dir}/{vid}.tif") and os.path.exists(f"{train_lbl_dir}/{vid}.tif"):
        sample_id = vid
        break

print(f"Using sample: {sample_id}")

volume_raw = tifffile.imread(f"{train_img_dir}/{sample_id}.tif")
gt_label = tifffile.imread(f"{train_lbl_dir}/{sample_id}.tif")
print(f"Volume shape: {volume_raw.shape}, dtype: {volume_raw.dtype}")
print(f"Label shape: {gt_label.shape}, unique: {np.unique(gt_label)}")

In [ ]:
# Preprocess
def val_transformation(image):
    data = {"image": image}
    pipeline = Compose([
        NormalizeIntensity(keys=["image"], nonzero=True, channel_wise=False),
    ])
    return pipeline(data)["image"]

volume = volume_raw.astype(np.float32)[None, ..., None]
volume = val_transformation(volume)
print(f"Preprocessed volume shape: {volume.shape}")

In [ ]:
# Load model
model = TransUNet(
    input_shape=(160, 160, 160, 1),
    encoder_name='seresnext50',
    classifier_activation=None,
    num_classes=3,
)
model.load_weights(
    "/kaggle/input/vsd-model/keras/transunet/3/transunet.seresnext50.160px.comboloss.weights.h5"
)
print(f"Model params: {model.count_params()/1e6:.1f}M")

swi = SlidingWindowInference(
    model, num_classes=3, roi_size=(160,160,160),
    sw_batch_size=1, mode='gaussian', overlap=0.1,
)

## Run Inference (TTA) — Save Intermediate Outputs

In [ ]:
def predict_with_tta_full(inputs, swi):
    """Return both fg_prob (float) and raw argmax (int)."""
    logits = []
    logits.append(swi(inputs))
    for axis in [1, 2, 3]:
        img_f = np.flip(inputs, axis=axis)
        p = swi(img_f)
        p = np.flip(p, axis=axis)
        logits.append(p)
    for k in [1, 2, 3]:
        img_r = np.rot90(inputs, k=k, axes=(2, 3))
        p = swi(img_r)
        p = np.rot90(p, k=-k, axes=(2, 3))
        logits.append(p)

    mean_logits = np.mean(logits, axis=0)
    mean_prob = ops.softmax(mean_logits, axis=-1)

    raw_pred = np.array(mean_prob).argmax(-1).astype(np.uint8).squeeze()
    fg_prob = np.array(mean_prob[0, ..., 1]).astype(np.float32)
    return fg_prob, raw_pred

fg_prob, raw_pred = predict_with_tta_full(volume, swi)
print(f"fg_prob shape: {fg_prob.shape}, range: [{fg_prob.min():.3f}, {fg_prob.max():.3f}]")
print(f"raw_pred shape: {raw_pred.shape}, classes: {np.unique(raw_pred)}")

In [ ]:
# Post-processing (0.549 recipe)
def build_anisotropic_struct(z_radius, xy_radius):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0: return None
    if z == 0 and r > 0:
        size = 2*r+1; struct = np.zeros((1,size,size), dtype=bool); cy=cx=r
        for dy in range(-r,r+1):
            for dx in range(-r,r+1):
                if dy*dy+dx*dx<=r*r: struct[0,cy+dy,cx+dx]=True
        return struct
    if z > 0 and r == 0:
        struct = np.zeros((2*z+1,1,1), dtype=bool); struct[:,0,0]=True; return struct
    depth=2*z+1; size=2*r+1; struct=np.zeros((depth,size,size), dtype=bool); cz=z; cy=cx=r
    for dz in range(-z,z+1):
        for dy in range(-r,r+1):
            for dx in range(-r,r+1):
                if dy*dy+dx*dx<=r*r: struct[cz+dz,cy+dy,cx+dx]=True
    return struct

def topo_postprocess(fg_prob, T_low=0.15, T_high=0.50, z_radius=3, xy_radius=1, dust_min_size=150):
    strong = fg_prob >= T_high
    weak = fg_prob >= T_low
    if not strong.any(): return np.zeros_like(fg_prob, dtype=np.uint8)
    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)
    if not mask.any(): return np.zeros_like(fg_prob, dtype=np.uint8)
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)
    return mask.astype(np.uint8)

pp_pred = topo_postprocess(fg_prob)
print(f"pp_pred shape: {pp_pred.shape}, fg voxels: {pp_pred.sum():,}")
print(f"GT class-1 voxels: {(gt_label==1).sum():,}")

In [ ]:
# Save intermediates as .npy for local visualization later
npy_dir = os.path.join(out_root, "npy")
os.makedirs(npy_dir, exist_ok=True)
np.save(os.path.join(npy_dir, "fg_prob.npy"), fg_prob)
np.save(os.path.join(npy_dir, "raw_pred.npy"), raw_pred)
np.save(os.path.join(npy_dir, "pp_pred.npy"), pp_pred)
np.save(os.path.join(npy_dir, "gt_label.npy"), gt_label)
print(f"Saved .npy files to {npy_dir}")

---
## Viz 1: Probability Heatmaps

In [ ]:
n_slices = 8
D = fg_prob.shape[0]
indices = np.linspace(0, D - 1, n_slices, dtype=int)

fig, axes = plt.subplots(2, n_slices, figsize=(3 * n_slices, 6))

for i, z in enumerate(indices):
    im = axes[0, i].imshow(fg_prob[z], cmap="inferno", vmin=0, vmax=1)
    axes[0, i].set_title(f"z={z}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(fg_prob[z], cmap="inferno", vmin=0, vmax=1)
    gt_slice = (gt_label[z] == 1).astype(float)
    if gt_slice.any():
        axes[1, i].contour(gt_slice, levels=[0.5], colors="lime", linewidths=0.8)
    axes[1, i].set_title(f"z={z} + GT", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("P(surface)", fontsize=10)
axes[1, 0].set_ylabel("P + GT contour", fontsize=10)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label="P(surface)")
fig.suptitle(f"Probability Heatmaps — sample {sample_id}", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(out_root, "prob_heatmaps.png"), dpi=150, bbox_inches="tight")
plt.show()

## Viz 2: Before/After Post-Processing vs GT

In [ ]:
cmap3 = ListedColormap(["black", "white", "gray"])

fig, axes = plt.subplots(3, n_slices, figsize=(3 * n_slices, 9))

for i, z in enumerate(indices):
    axes[0, i].imshow(raw_pred[z], cmap=cmap3, vmin=0, vmax=2)
    axes[0, i].set_title(f"z={z}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(pp_pred[z], cmap="gray", vmin=0, vmax=1)
    axes[1, i].axis("off")

    axes[2, i].imshow(gt_label[z], cmap=cmap3, vmin=0, vmax=2)
    axes[2, i].axis("off")

axes[0, 0].set_ylabel("Raw argmax", fontsize=11)
axes[1, 0].set_ylabel("Post-processed", fontsize=11)
axes[2, 0].set_ylabel("Ground truth", fontsize=11)
fig.suptitle(f"Before/After PP vs GT — sample {sample_id}", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(out_root, "pp_compare.png"), dpi=150, bbox_inches="tight")
plt.show()

## Viz 3: Diff Maps — What Did PP Change?

In [ ]:
raw_bin = (raw_pred == 1).astype(np.uint8)
pp_bin = pp_pred.astype(np.uint8)
gt_bin = (gt_label == 1).astype(np.uint8)

added = (pp_bin == 1) & (raw_bin == 0)
removed = (pp_bin == 0) & (raw_bin == 1)

fig, axes = plt.subplots(2, n_slices, figsize=(3 * n_slices, 6))

for i, z in enumerate(indices):
    rgb = np.zeros((*raw_bin.shape[1:], 3), dtype=np.float32)

    unchanged_fg = (pp_bin[z] == 1) & (raw_bin[z] == 1)
    rgb[unchanged_fg] = [1, 1, 1]            # white

    rgb[added[z] & (gt_bin[z] == 1)] = [0, 1, 0]       # green: added correct
    rgb[added[z] & (gt_bin[z] == 0)] = [1, 0, 0]       # red: added wrong
    rgb[removed[z] & (gt_bin[z] == 0)] = [0.3, 0.5, 1] # blue: removed correct
    rgb[removed[z] & (gt_bin[z] == 1)] = [1, 0.6, 0]   # orange: removed wrong

    axes[0, i].imshow(rgb)
    axes[0, i].set_title(f"z={z}", fontsize=9)
    axes[0, i].axis("off")

    axes[1, i].imshow(gt_bin[z], cmap="gray", vmin=0, vmax=1)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("PP diff", fontsize=11)
axes[1, 0].set_ylabel("GT (class 1)", fontsize=11)

legend_elements = [
    Patch(facecolor="white", edgecolor="gray", label="Unchanged FG"),
    Patch(facecolor="green", label="Added (correct)"),
    Patch(facecolor="red", label="Added (wrong)"),
    Patch(facecolor="cornflowerblue", label="Removed (correct)"),
    Patch(facecolor="orange", label="Removed (wrong)"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=5, fontsize=9)
fig.suptitle(f"PP Diff Map — sample {sample_id}", fontsize=14)
plt.tight_layout(rect=[0, 0.05, 1, 0.97])
fig.savefig(os.path.join(out_root, "diff_map.png"), dpi=150, bbox_inches="tight")
plt.show()

# Stats
n_added = added.sum()
n_removed = removed.sum()
n_added_ok = (added & (gt_bin == 1)).sum()
n_removed_ok = (removed & (gt_bin == 0)).sum()
print(f"PP added {n_added:,} voxels ({n_added_ok:,} correct, {n_added - n_added_ok:,} wrong)")
print(f"PP removed {n_removed:,} voxels ({n_removed_ok:,} correct, {n_removed - n_removed_ok:,} wrong)")

## Viz 4: 3D Surface Renders

In [ ]:
def render_surface(mask, ax, color, alpha, downsample=0.4):
    vol = zoom(mask.astype(np.float32), downsample, order=1)
    try:
        verts, faces, _, _ = measure.marching_cubes(vol, level=0.5, step_size=1, allow_degenerate=False)
    except (ValueError, RuntimeError):
        return 0
    verts = verts / downsample
    mesh = Poly3DCollection(verts[faces], alpha=alpha, linewidths=0.1)
    mesh.set_facecolor(color)
    mesh.set_edgecolor((*plt.cm.colors.to_rgb(color), 0.05))
    ax.add_collection3d(mesh)
    return len(faces)

shape = pp_pred.shape

# Side by side
fig = plt.figure(figsize=(16, 7))

ax1 = fig.add_subplot(1, 2, 1, projection="3d")
n1 = render_surface(pp_bin, ax1, "royalblue", 0.5)
ax1.set_xlim(0, shape[0]); ax1.set_ylim(0, shape[1]); ax1.set_zlim(0, shape[2])
ax1.set_title(f"Prediction ({n1:,} tri)", fontsize=12)
ax1.view_init(elev=25, azim=45)

ax2 = fig.add_subplot(1, 2, 2, projection="3d")
n2 = render_surface(gt_bin, ax2, "forestgreen", 0.5)
ax2.set_xlim(0, shape[0]); ax2.set_ylim(0, shape[1]); ax2.set_zlim(0, shape[2])
ax2.set_title(f"Ground Truth ({n2:,} tri)", fontsize=12)
ax2.view_init(elev=25, azim=45)

fig.suptitle(f"3D Surface — sample {sample_id}", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(out_root, "3d_side_by_side.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Overlay from multiple angles
fig, axes_arr = plt.subplots(1, 4, figsize=(20, 5), subplot_kw={"projection": "3d"})
for i, az in enumerate([0, 90, 180, 270]):
    ax = axes_arr[i]
    render_surface(gt_bin, ax, "forestgreen", 0.3)
    render_surface(pp_bin, ax, "royalblue", 0.3)
    ax.set_xlim(0, shape[0]); ax.set_ylim(0, shape[1]); ax.set_zlim(0, shape[2])
    ax.view_init(elev=20, azim=az)
    ax.set_title(f"azim={az}", fontsize=10)
fig.suptitle(f"Overlay: Pred (blue) vs GT (green) — sample {sample_id}", fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(out_root, "3d_overlay_angles.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print(f"\nAll outputs saved to {out_root}/")
!ls -la {out_root}/